# DSWP buzz-level analysis

This notebook reproduces the primary event-level results from publication-safe derived data. It never reads audio, raw tag channels, locations, animal names, or field identifiers. The distributed event table contains anonymous grouping tokens and summaries computed upstream from the reconstructed trajectories.

The notebook is an annotated interface to tested functions in `src/dswp/`; `scripts/run_pipeline.py` is the authoritative batch runner.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
sys.path.insert(0, str(ROOT / "src"))

from dswp.classification import attach_labels, classifier_robustness, gmm_model_selection
from dswp.comebacks import attach_comeback_context, detect_comeback_events, stratified_comeback_association
from dswp.energetics import event_energy_from_trajectories, paired_individual_energy
from dswp.io import load_buzz_events
from dswp.statistics import (
    compare_groups,
    depth_mixed_model,
    markov_leave_subset_out,
    pause_distribution_tests,
    roll_speed_bin_tests,
)
from dswp.trajectory_features import buzz_straightness

EVENTS = ROOT / "data/dswp/buzz_events.csv"
TRAJECTORY_INDEX = ROOT / "data/dswp/trajectory_index.csv"
COMEBACK_INDEX = ROOT / "data/dswp/comeback_trajectory_index.csv"

## 1. Classify recurrent buzz tactics

The main DSWP analysis fits the two-component Gaussian mixture model to quantile-normalized ramp-up duration and mean reconstructed speed. Posterior probabilities below 0.75 remain deliberately unclassified (`?`). Component names are assigned after fitting: the lower-speed component is A and the higher-speed component is B.

In [ ]:
events = attach_labels(load_buzz_events(EVENTS), method="gmm")
counts = events["buzz_type"].value_counts().reindex(["A", "B", "?"])
counts.rename("n_buzzes").to_frame()

The registered production result is A = 1,346, B = 1,142, and uncertain = 1,198 buzzes. A different count indicates a changed input table, software environment, or classifier implementation and should be investigated before figures are regenerated.

Production note: these reported labels use the original trajectory-summary speed field, while the selected pitch-adaptive reconstruction speed is retained separately for trajectory-derived analyses.

### Model selection and threshold-free classifier checks

The BIC minimum is `k=2` for the ramp-up/3-D-speed plane actually used by the paper and for max-click-rate/roll. The diagnostic ramp-plus-plateau-duration/median-click-rate plane instead prefers `k=3`; the old notebook calls this “total duration” even though it excludes the terminal segment. The public workflow reports the actual result rather than marking every curve as two-component. The second table reproduces the SI's threshold-free agreement and Cohen's κ values for GMM and rank-space corner alternatives.

In [ ]:
model_selection = gmm_model_selection(events)
model_optima = model_selection.loc[
    model_selection.groupby("feature_pair")["bic"].idxmin(),
    ["feature_pair", "n_components", "bic"],
]
robustness, robustness_confusion = classifier_robustness(events)
display(model_optima)
display(robustness)

## 2. Compare acoustic and kinematic features

The paper uses non-parametric A/B contrasts. The table below reports sample sizes, group medians, a two-sided Mann–Whitney U test, and Cliff's delta. Positive Cliff's delta means that values tend to be larger for A than for B.

In [ ]:
metrics = [
    "ramp_up_duration_s",
    "avg_speed_3d_m_s",
    "avg_vertical_speed_m_s",
    "avg_roll_speed_deg_s",
    "avg_depth_m",
    "median_click_rate_hz",
    "pre_buzz_vertical_velocity_m_s",
    "post_buzz_pause_s",
]
contrasts = pd.DataFrame([compare_groups(events, metric) for metric in metrics])
contrasts

### Nesting-aware depth, roll time course, and pause distribution

The headline depth result is not the buzz-level rank test alone. It is a REML mixed model with random intercepts for individual and dive nested within individual. The roll panel uses independent tests in each 2-s bin with Bonferroni correction across bins. The pause panel reports both Mann–Whitney and Kolmogorov–Smirnov tests on non-terminal pause durations.

In [ ]:
depth_result = depth_mixed_model(events)
roll_bins = pd.read_csv(ROOT / "data/dswp/roll_speed_bins.csv")
roll_tests = roll_speed_bin_tests(events, roll_bins)
pause_tests = pause_distribution_tests(events)

display(pd.Series(depth_result, name="depth_mixed_model"))
display(roll_tests)
display(pd.Series(pause_tests, name="pause_distribution"))

## 3. Quantify within-dive switching

Production note: buzzes are ordered within anonymous deployment–dive groups. Ambiguous buzzes remain as gaps: transitions touching `?` are discarded after counting, so a run A–?–B is not bridged into A→B. The figure is recomputed 200 times after excluding ten deployments per iteration and reports the mean ± s.d., matching the Methods.

In [ ]:
transitions = markov_leave_subset_out(events, n_resamples=200, seed=42)
transitions.sort_values(["from_state", "to_state"])

## 4. Recompute trajectory-centred straightness

Straightness is recomputed from the released numeric trajectories rather than trusted as a copied summary. Each path is resampled at 1 m spatial intervals. Before-buzz windows end at the first click; after-buzz windows start at the last click.

In [ ]:
straightness = buzz_straightness(events, TRAJECTORY_INDEX, windows_m=(20, 30, 40))
straightness_with_labels = events[["event_id", "buzz_type"]].merge(
    straightness, on="event_id", validate="one_to_one"
)
straightness_tests = pd.DataFrame([
    compare_groups(straightness_with_labels, f"straightness_{phase}_{distance}m")
    for phase in ("before", "after")
    for distance in (20, 30, 40)
])
straightness_tests

The expected two-sided P value for 20 m post-buzz straightness is approximately 0.40. This is the value aligned with the current Figure 4 caption.

## 5. Recompute the result-producing energetic estimate

The calculation below uses speeds obtained from the pitch-adaptive reconstructed trajectories over the full first-click–last-click buzz interval. It integrates parasite-drag work and positive tangential-acceleration work, then adds resting metabolism. Within-whale A and B means are computed first, followed by the mean across whales.

This implementation reproduces the reported 1.28 MJ and 9.07 MJ means and matches the current Methods and Supplementary Information. A step-by-step component derivation and the Aoki directional comparison are provided in `02_energy_model_derivation.ipynb`.

In [ ]:
energy = event_energy_from_trajectories(events, TRAJECTORY_INDEX)
per_individual = paired_individual_energy(energy, individual_col="individual_id")
summary_mj = per_individual[["A", "B"]].mean().div(1e6).rename("mean_MJ")
pd.concat([summary_mj, per_individual[["A", "B"]].count().rename("n_individuals")], axis=1)

## 6. Reproduce the spatial-recurrence association

The recurrence analysis needs a deployment-scale coordinate frame, so it uses a separate 20-s subsample of the reconstructed deployment paths. The result-producing notebook used a 3-D 50/150-m return rule, grouped return samples within 120 s, retained events deeper than 200 m, and buffered event intervals by 120 s before matching buzz starts.

Production note: the manuscript label table reproduces OR 1.11. Applying the current classifier to the same recurrence events gives OR 1.15 as a separately labelled sensitivity; both associations are null.

In [ ]:
comeback_events, comeback_deployments = detect_comeback_events(COMEBACK_INDEX)
comeback_context = attach_comeback_context(events, comeback_events).merge(
    pd.read_csv(ROOT / "data/dswp/comeback_manuscript_labels.csv"),
    on="event_id",
    validate="one_to_one",
)

association_rows = []
for version, label_column in [
    ("manuscript", "manuscript_buzz_type"),
    ("current_classifier", "buzz_type"),
]:
    _, summary = stratified_comeback_association(
        comeback_context, label_col=label_column
    )
    association_rows.append({"version": version, **summary})
pd.DataFrame(association_rows)

## Reproducible outputs

The notebook leaves inputs untouched. To regenerate every machine-readable table and figure in one controlled pass, run `python scripts/run_pipeline.py` from the repository root. The batch runner audits the release boundary before and after analysis and writes its headline checks to `results/pipeline_summary.json`.